# Phase 3: Feature Engineering

Loads the processed data from Phase 2, builds TF-IDF and CountVectorizer
feature matrices, splits into train/test sets, and saves everything needed
for Phase 4 (Modeling & Evaluation).

In [1]:
import pandas as pd
import joblib
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

df = pd.read_pickle("../data/processed_postings.pkl")
print(f"Loaded {df.shape[0]} processed postings")

Loaded 17880 processed postings


## TF-IDF and CountVectorizer

Both use unigrams + bigrams, capped at 5,000 features, with a minimum
document frequency of 3 (drops noise from one-off typos/rare tokens). We
build both so Phase 4 can directly compare their effect on model performance.

In [2]:
X_text = df["full_text_clean"]
y = df["fraudulent"]

tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), min_df=3, stop_words="english")
X_tfidf = tfidf.fit_transform(X_text)

count_vec = CountVectorizer(max_features=5000, ngram_range=(1, 2), min_df=3, stop_words="english")
X_count = count_vec.fit_transform(X_text)

print(f"TF-IDF feature matrix shape: {X_tfidf.shape}")
print(f"CountVectorizer feature matrix shape: {X_count.shape}")

TF-IDF feature matrix shape: (17880, 5000)
CountVectorizer feature matrix shape: (17880, 5000)


## Train/Test Split

A stratified 80/20 split preserves the ~4.84% fraud rate in both sets.

In [3]:
X_train_tfidf, X_test_tfidf, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)
X_train_count, X_test_count, _, _ = train_test_split(
    X_count, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"Train size: {X_train_tfidf.shape[0]}  |  Test size: {X_test_tfidf.shape[0]}")
print(f"Train fraud rate: {y_train.mean()*100:.2f}%  |  Test fraud rate: {y_test.mean()*100:.2f}%")

Train size: 14304  |  Test size: 3576
Train fraud rate: 4.84%  |  Test fraud rate: 4.84%


## Save everything for Phase 4

Bundled into one file so Phase 4 only needs one load call. Includes both
vectorizers (needed later to save the final TF-IDF vectorizer for the demo app).

In [4]:
features_bundle = {
    "X_train_tfidf": X_train_tfidf, "X_test_tfidf": X_test_tfidf,
    "X_train_count": X_train_count, "X_test_count": X_test_count,
    "y_train": y_train, "y_test": y_test,
    "tfidf_vectorizer": tfidf, "count_vectorizer": count_vec,
}
joblib.dump(features_bundle, "../data/features_bundle.joblib")
print("Saved features_bundle.joblib for Phase 4")

Saved features_bundle.joblib for Phase 4
